## Set up requirements

In [ ]:
from pathlib import Path
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nexus_continuous.envs.env_registry import ENV_REGISTRY

from nexus_continuous.llm.pipeline import generate_skillset
from nexus_continuous.llm.client import LLMClient, LLMConfig
from nexus_continuous.llm.interpreter import make_policy_module

from scripts.run_llm_experiment import run_llm_experiment
from scripts.run_llm_comparison import compare_policies

## Available Environments

In [ ]:
# Expects a dictionary output of environemtns
ENV_REGISTRY.keys()

In [ ]:
# Choose environments 
ENV = "WalkerWalk"

In [ ]:
# Inspect environment metadata 
env = ENV_REGISTRY[ENV]

print(env["task"])
print()
print("Observation fields")
for f in env["fields"]:
    print(" -", f)

## Skills Generation

In [ ]:
# Generate skills 
client = LLMClient(
    LLMConfig(
        backend = "hf"
    )
)

skillset = generate_skillset(
    env_name = ENV,
    observation_schema = "\n".join(env["fields"]),
    task_description = ["task"],
    client = client
)

from pprint import pprint
pprint(skillset)

In [ ]:
# Save JSON 
from nexus_continuous.llm.pipeline import save_skillset

save_skillset(
    skillset,
    f"results/{ENV}/skillset.json"
)

In [ ]:
# Skill hierarchy 
names = [s.name for s in skillset.skills]

plt.figure(figsize = (8,3))
plt.plot(
    range(len(names)),
    np.ones(len(names)),
    "-o"
)
plt.yticks([])
plt.xticks(
    range(len(names)),
    names,
    rotation = 25
)
plt.title("Generated Skill Progression")
plt.show()

## Reward Structure 

In [ ]:
rows = []
for skill in skillset.skills:
    for reward in skill.reward_terms:
        rows.append({
            "Skill":skill.name,
            "Reward": reward.type,
            "Weight": reward.weight,
            "LHS": reward.lhs,
            "Threshold": reward.threshold
        })
pd.DataFrame(rows)

## Policy compiling

In [ ]:
from dataclasses import asdict 

policy = make_policy_module(asdict(skillset))

## Run LLM experiment

In [ ]:
metrics = run_llm_experiment(
    env_name = ENV,
    config = "configs/walker_walk_nesy.yaml",
    backend = "hf",
    seed = 0
)

In [ ]:
# Compare against handwritten 
comparison = compare_policies(
    env_name = ENV,
    config = "configs/walker_walk_nesy.yaml",
    backend = "hf",
    num_seeds = 3
)

In [ ]:
# Comparison table
pd.DataFrame([comparison])

In [ ]:
# Comparison plot
plt.figure(figsize = (5,4))
plt.bar(
    ["Handwritten", "LLM"],
    [
        comparison["hand_mean"],
        comparison["llm_mean"]
    ]
)
plt.ylabel("Mean Return")
plt.title(ENV)
plt.show()

In [ ]:
# Multi-environment experiment 
results = []
for env in ENV_REGISTRY:
    try:
        row = compare_policies(
            env,
            config = "configs/walker_walk_nesy.yaml",
            backend = "hf",
            num_seeds = 3
        )
        results.append(row)
    except Exception as e:
        print(env, e)

In [ ]:
# Results dataframe 
df = pd.DataFrame(results)

## Overall comparison 

In [ ]:
plt.figure(figsize = (9,4))
x = np.arange(len(df))
width = 0.35
plt.bar(
    x-width/2,
    df.hand_mean,
    width,
    label = "Handwritten"
)
plt.bar(
    x+width/2,
    df.llm_mean,
    width,
    label = "LLM"
)
plt.xticks(
    x,
    df.env,
    rotation = 30
)
plt.ylabel("Mean Return")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save results 
Path("results").mkdir(exist_ok = True)
df.to_csv(
    "results/comparison.csv",
    index = False
)